In [1]:
import pandas as pd
import numpy as np
import faiss
import os, textwrap, warnings
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH  = "../data/processed/severity_data.csv"
EMB_PATH   = "../data/processed/embeddings.npy"
INDEX_PATH = "../models/faiss_index.bin"
os.makedirs("../models", exist_ok=True)

TOP_K       = 10
EMBED_MODEL = "all-MiniLM-L6-v2"

print("Setup complete")

Setup complete


In [2]:
df = pd.read_csv(DATA_PATH)
embeddings_full = np.load(EMB_PATH).astype("float32")

PROCESSED_PATH = "../data/processed/processed_data.csv"
proc = (pd.read_csv(PROCESSED_PATH)[["text_clean"]]
        .reset_index()
        .rename(columns={"index": "_emb_idx"}))

proc = proc.drop_duplicates(subset="text_clean", keep="first")

df = (df.merge(proc, on="text_clean", how="left")
        .dropna(subset=["_emb_idx"])
        .reset_index(drop=True))
df["_emb_idx"] = df["_emb_idx"].astype(int)

embeddings = embeddings_full[df["_emb_idx"].values]
df = df.drop(columns=["_emb_idx"])

assert len(df) == embeddings.shape[0], \
    f"Row mismatch: df={len(df)}, embeddings={embeddings.shape[0]}"

print(f"Loaded: {len(df):,} complaints | embeddings: {embeddings.shape}")
print(f"Embedding dim: {embeddings.shape[1]}")

Loaded: 17,181 complaints | embeddings: (17181, 384)
Embedding dim: 384


In [3]:
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(embeddings)
index.add(embeddings)

faiss.write_index(index, INDEX_PATH)
print(f"FAISS index built: {index.ntotal:,} vectors | saved to {INDEX_PATH}")

FAISS index built: 17,181 vectors | saved to ../models/faiss_index.bin


In [4]:
encoder = SentenceTransformer(EMBED_MODEL)

def encode_query(query: str) -> np.ndarray:
    vec = encoder.encode([query], normalize_embeddings=True).astype("float32")
    return vec


def retrieve(query: str, k: int = TOP_K) -> pd.DataFrame:
    q_vec   = encode_query(query)
    scores, indices = index.search(q_vec, k)

    results = df.iloc[indices[0]].copy()
    results["similarity"] = scores[0]
    return results[[
        "similarity", "cluster", "sentiment", "severity_score",
        "product", "issue", "company_response", "narrative"
    ]].reset_index(drop=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2999.65it/s]


In [5]:
def build_prompt(query: str, retrieved: pd.DataFrame) -> str:
    context_blocks = []
    for i, row in retrieved.iterrows():
        context_blocks.append(
            f"[Complaint {i+1}]\n"
            f"Product: {row['product']} | Issue: {row['issue']}\n"
            f"Sentiment: {row['sentiment']} | Severity: {row['severity_score']}\n"
            f"Resolution: {row['company_response']}\n"
            f"Narrative: {row['narrative'][:400]}\n"
        )
    context = "\n".join(context_blocks)

    return (
        f"You are a CFPB complaint analysis assistant. "
        f"Answer the following question using ONLY the provided complaints as evidence. "
        f"Be concise and cite specific complaint numbers when relevant.\n\n"
        f"QUESTION: {query}\n\n"
        f"COMPLAINTS:\n{context}\n\n"
        f"ANSWER:"
    )


def synthesize(query: str, retrieved: pd.DataFrame) -> str:
    prompt = build_prompt(query, retrieved)

    # Anthropic Claude
    # client = anthropic.Anthropic()
    # msg = client.messages.create(
    #     model="claude-sonnet-4-20250514",
    #     max_tokens=512,
    #     messages=[{"role": "user", "content": prompt}]
    # )
    # return msg.content[0].text

    # OpenAI GPT-4o
    # client = OpenAI()
    # resp = client.chat.completions.create(
    #     model="gpt-4o",
    #     messages=[{"role": "user", "content": prompt}],
    #     max_tokens=512,
    # )
    # return resp.choices[0].message.content

    # Local HF (no API key)
    # pipe = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.2")
    # return pipe(prompt, max_new_tokens=256)[0]["generated_text"].split("ANSWER:")[-1].strip()

    lines = [f"Top {len(retrieved)} retrieved complaints for: '{query}'"]
    for i, row in retrieved.iterrows():
        lines.append(
            f"  [{i+1}] Similarity={row['similarity']:.3f} "
            f"Cluster={row['cluster']} Sentiment={row['sentiment']}\n"
            f"       {row['narrative'][:200]}"
        )
    return "\n".join(lines)

In [6]:
def ask(query: str, k: int = TOP_K, verbose: bool = True) -> str:
    retrieved = retrieve(query, k=k)

    if verbose:
        print(f"QUERY: {query}")
        print(f"Top-{k} retrieved complaints:")
        for i, row in retrieved.iterrows():
            print(f"  [{i+1}] sim={row['similarity']:.3f} "
                  f"cluster={row['cluster']} sent={row['sentiment']} "
                  f"sev={row['severity_score']}")
            print(f"       {row['narrative'][:150]}")

    answer = synthesize(query, retrieved)

    if verbose:
        print("ANSWER:")
        print(textwrap.fill(answer, width=80, initial_indent="  ",
                            subsequent_indent="  "))

    return answer


example_queries = [
    "Show me cases where customers disputed a credit report and got no relief.",
    "What are the most common complaints about debt collection practices?",
    "Find high-severity card payment complaints where the company closed without relief.",
    "Which product category has the most negative sentiment complaints?",
]

for q in example_queries:
    ask(q, k=5)

QUERY: Show me cases where customers disputed a credit report and got no relief.
Top-5 retrieved complaints:
  [1] sim=0.662 cluster=0 sent=positive sev=3
       I am writing to formally dispute the reporting of a charge-off account on my credit report that I believe contains multiple inaccuracies. The account 
  [2] sim=0.651 cluster=0 sent=negative sev=3
       How are you? Im extremely upset and shocked about the way this charge-off account is reporting on my credit report which continues to report with inco
  [3] sim=0.641 cluster=0 sent=negative sev=3
       I am disputing several accounts that are inaccurately reported. I request that the following accounts be immediately investigated and corrected to ref
  [4] sim=0.636 cluster=0 sent=negative sev=4
       I have been disputing fraud ITEMS on my credit report. The items are not showing in dispute nor are they removed from my report. The bureaus seem to i
  [5] sim=0.636 cluster=0 sent=positive sev=3
       I am writing to disput

In [7]:
def batch_query(queries: list[str], k: int = TOP_K) -> pd.DataFrame:
    rows = []
    for q in queries:
        retrieved = retrieve(q, k=k)
        answer    = synthesize(q, retrieved)
        rows.append({
            "query":           q,
            "top1_similarity": retrieved["similarity"].iloc[0],
            "top1_cluster":    retrieved["cluster"].iloc[0],
            "top1_sentiment":  retrieved["sentiment"].iloc[0],
            "top1_severity":   retrieved["severity_score"].iloc[0],
            "answer_preview":  answer[:200],
        })
    return pd.DataFrame(rows)


results = batch_query(example_queries)
print(results.to_string(index=False))

results.to_csv("../outputs/reports/rag_query_results.csv", index=False)
print("Saved: ../outputs/reports/rag_query_results.csv")

                                                                              query  top1_similarity  top1_cluster top1_sentiment  top1_severity                                                                                                                                                                                             answer_preview
          Show me cases where customers disputed a credit report and got no relief.         0.661921             0       positive              3 Top 10 retrieved complaints for: 'Show me cases where customers disputed a credit report and got no relief.'\n  [1] Similarity=0.662 Cluster=0 Sentiment=positive\n       I am writing to formally dispute
               What are the most common complaints about debt collection practices?         0.667916             1       positive              3 Top 10 retrieved complaints for: 'What are the most common complaints about debt collection practices?'\n  [1] Similarity=0.668 Cluster=1 Sentiment=positive\n 

In [8]:
def retrieve_filtered(query: str,
                      cluster: int | None = None,
                      product: str | None = None,
                      sentiment: str | None = None,
                      min_severity: int | None = None,
                      k: int = TOP_K) -> pd.DataFrame:
    mask = pd.Series([True] * len(df))

    if cluster is not None:
        mask &= df["cluster"] == cluster
    if product is not None:
        mask &= df["product"].str.lower().str.contains(product.lower(), na=False)
    if sentiment is not None:
        mask &= df["sentiment"] == sentiment
    if min_severity is not None:
        mask &= df["severity_score"] >= min_severity

    sub_df  = df[mask].reset_index()
    sub_emb = embeddings[mask.values]

    if len(sub_df) == 0:
        print("No complaints match the given filters.")
        return pd.DataFrame()

    sub_index = faiss.IndexFlatIP(embeddings.shape[1])
    sub_emb_f = sub_emb.astype("float32").copy()
    faiss.normalize_L2(sub_emb_f)
    sub_index.add(sub_emb_f)

    q_vec = encode_query(query)
    scores, indices = sub_index.search(q_vec, min(k, len(sub_df)))

    results = sub_df.iloc[indices[0]].copy()
    results["similarity"] = scores[0]
    return results[["similarity", "cluster", "sentiment", "severity_score",
                    "product", "issue", "company_response", "narrative"]].reset_index(drop=True)


filtered = retrieve_filtered(
    query="consumers not notified before debt collection",
    cluster=1,
    sentiment="negative",
    min_severity=4,
    k=5,
)
print(f"Filtered results: {len(filtered)} complaints")
if not filtered.empty:
    for _, row in filtered.iterrows():
        print(f"  sim={row['similarity']:.3f} sev={row['severity_score']} "
              f"{row['narrative'][:150]}")

Filtered results: 5 complaints
  sim=0.408 sev=4 I have not supplied proof under the doctrine of estoppel by the silence XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX  Presume that no proof of the alleged 
  sim=0.403 sev=4 I have not supplied under the doctrine of estoppel by the silence of Engelhardt v Gravens ( mo ) 281 SW715,719, I presume that no proof of the alleged
  sim=0.380 sev=4 In accordance with the Fair Credit Reporting Act ; XXXX XXXX XXXX account # XXXX ; XXXX/XXXX  account # XXXX ; XXXX XXXX  XXXX  account # XXXX XXXX XX
  sim=0.373 sev=4 Comenity Capital Bank, which is part of Bread Financial. 
XXXX  XXXX XXXX XXXX  are issued by Comenity Capital Bank. XXXX ( TDD/TTY : XXXX ) obsessive
  sim=0.369 sev=4 There are multi accounts that do n't belong to me, I have contacted the them directly and they were very rude, I have disputed them with all XXXX cred
